# UMUD Challenge — segmentation + geometry pipeline

**Pipeline**
1. Find the ultrasound field of view (FOV) in each image and crop away scanner UI.
2. Train two U-Nets (ResNet-34 encoder): one for aponeuroses, one for fascicle fragments.
3. Fit the superficial / deep aponeuroses as robust lines, extract fascicle segments between them.
4. Compute PA (angle to deep aponeurosis), MT (perpendicular distance at 3 positions) and FL
   (linear extrapolation between the aponeuroses at 3 anchor positions) — mirroring the manual protocol.
5. Pixel → mm calibration from the depth-scale tick marks on the screen.
6. Temporal smoothing across near-identical video frames, then write `submission.csv`.



**Challenge-specific design choices**
- *Metric:* UMUD Score = mean(MAE_PA/6°, MAE_FL/12 mm, MAE_MT/3 mm). MAE rewards medians, so every
  aggregation step uses medians; 1 mm of MT error costs as much as 2° of PA, so the MT edge definition matters.
- *Duplicates:* byte-identical image–mask pairs (670 in the fascicle set, per the discussion thread) are removed
  via SHA-256; near-duplicates (video frames) are kept but forced into the same fold.
- *Reproducibility rule:* the whole pipeline re-runs on new data with no leaderboard input. Ensembling is only
  across fold models trained here (`N_FOLDS`); no leaderboard scores or LB-derived weights are used anywhere.
- *Novelty vs. DL_Track_US:* ResNet-34 U-Net (DL_Track uses VGG16), protocol-matched geometric read-out
  (3-position MT/FL, deep-aponeurosis PA), automatic screen-ruler calibration and temporal frame smoothing.

In [1]:
import os, re, json, math, time, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

CFG = dict(
    INPUT_ROOT="/kaggle/input",
    WORK_DIR="/kaggle/working",
    SEED=42,
    IMG_SIZE=512,           # network input (square)
    VAL_FRAC=0.15,
    BATCH=8,
    LR=3e-4,
    WD=1e-4,
    APO_EPOCHS=40,
    FASC_EPOCHS=30,
    NUM_WORKERS=2,
    TRAIN=True,             # False -> load weights from WEIGHTS_DIR
    WEIGHTS_DIR=None,       # e.g. "/kaggle/input/my-umud-weights"; None -> WORK_DIR
    PRETRAINED=True,        # ImageNet encoder (needs internet once; falls back to random init)
    FASC_DILATE=3,          # thicken thin fascicle labels for training
    TTA=True,               # horizontal-flip test-time augmentation
    APO_THR=0.5,
    FASC_THR=0.35,
    N_FOLDS=1,              # >1 -> group k-fold models, probabilities averaged (reproducible ensemble)
    MT_EDGE="inner",        # "inner": inner aponeurosis borders (manual protocol); "center": band centre lines
    PA_SHRINK_K=2.0,        # low-confidence PA is shrunk toward the group median: w = n_segs/(n_segs+K)
    LOCAL_LABELS=None,      # optional CSV (image_id, pa_deg, fl_mm, mt_mm) of your own annotations to score against
    MIN_FASC_LEN_FRAC=0.04, # min fascicle-segment length as fraction of FOV width
    PA_RANGE=(5.0, 45.0),
    FL_RANGE=(30.0, 200.0),
    MT_RANGE=(10.0, 50.0),
    # calibration
    TICK_MM_CANDIDATES=(2.0, 5.0, 10.0),  # possible mm per tick interval
    MT_PRIOR_MM=21.0,       # typical MT used only to resolve tick-unit ambiguity per group
    DEFAULT_DEPTH_MM=50.0,  # used only when no ticks are found anywhere in a group
    CALIB_MM_PER_TICK={},   # manual override per image size, e.g. {"1551x813": 5.0}
    CALIB_PX_PER_MM={},     # manual override per image size, e.g. {"1024x768": 12.3}
    # temporal smoothing
    SEQ_SIM_THR=0.93,
    SEQ_MAX_NEIGHBORS=4,
    DEBUG_MAX_TRAIN=None,   # int -> subsample training sets for a quick dry run
)
# optional JSON overrides (handy for quick experiments / CI)
if os.environ.get("UMUD_CFG"):
    CFG.update(json.loads(os.environ["UMUD_CFG"]))

random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
WORK = Path(CFG["WORK_DIR"]); WORK.mkdir(parents=True, exist_ok=True)
WEIGHTS = Path(CFG["WEIGHTS_DIR"]) if CFG["WEIGHTS_DIR"] else WORK
print(json.dumps({k: v for k, v in CFG.items() if not isinstance(v, dict)}, indent=1))

{
 "INPUT_ROOT": "/kaggle/input",
 "WORK_DIR": "/kaggle/working",
 "SEED": 42,
 "IMG_SIZE": 512,
 "VAL_FRAC": 0.15,
 "BATCH": 8,
 "LR": 0.0003,
 "WD": 0.0001,
 "APO_EPOCHS": 40,
 "FASC_EPOCHS": 30,
 "NUM_WORKERS": 2,
 "TRAIN": true,
 "WEIGHTS_DIR": null,
 "PRETRAINED": true,
 "FASC_DILATE": 3,
 "TTA": true,
 "APO_THR": 0.5,
 "FASC_THR": 0.35,
 "N_FOLDS": 1,
 "MT_EDGE": "inner",
 "PA_SHRINK_K": 2.0,
 "LOCAL_LABELS": null,
 "MIN_FASC_LEN_FRAC": 0.04,
 "PA_RANGE": [
  5.0,
  45.0
 ],
 "FL_RANGE": [
  30.0,
  200.0
 ],
 "MT_RANGE": [
  10.0,
  50.0
 ],
 "TICK_MM_CANDIDATES": [
  2.0,
  5.0,
  10.0
 ],
 "MT_PRIOR_MM": 21.0,
 "DEFAULT_DEPTH_MM": 50.0,
 "SEQ_SIM_THR": 0.93,
 "SEQ_MAX_NEIGHBORS": 4,
 "DEBUG_MAX_TRAIN": null
}


## 1. Data discovery and I/O helpers

In [2]:
IMG_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def list_images(d):
    return sorted(p for p in Path(d).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT)

def find_dirs(root):
    """Locate the competition folders regardless of version suffixes (_v1, _v2 ...)."""
    pats = [("apo_img", r"^apo_im"), ("apo_mask", r"^apo_mask"),
            ("fasc_img", r"^fasc_im"), ("fasc_mask", r"^fasc_mask"), ("test", r"^test")]
    hits = {}
    for d in sorted(Path(root).rglob("*")):
        if not d.is_dir():
            continue
        n = d.name.lower()
        for key, pat in pats:
            if key not in hits and re.match(pat, n):
                hits[key] = d
    return hits

def read_gray(p):
    im = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if im is None:
        from PIL import Image
        im = np.array(Image.open(p))
        if im.ndim == 3:
            im = im[..., ::-1]  # RGB -> BGR
    if im.ndim == 3:
        im = im[..., :3]
        im = cv2.cvtColor(np.ascontiguousarray(im), cv2.COLOR_BGR2GRAY)
    if im.dtype != np.uint8:
        im = im.astype(np.float32)
        im = (255 * (im - im.min()) / max(float(im.max() - im.min()), 1e-6)).astype(np.uint8)
    return im

def read_mask(p):
    m = read_gray(p)
    mx = m.max()
    return (m > (mx / 2.0 if mx > 0 else 1)).astype(np.uint8)

def norm_stem(p):
    s = Path(p).stem.lower()
    s = re.sub(r"[_\-\s]?(mask|label|seg|gt)s?$", "", s)
    s = re.sub(r"^(mask|label)s?[_\-\s]?", "", s)
    return s

def pair_images_masks(img_dir, mask_dir):
    imgs, masks = list_images(img_dir), list_images(mask_dir)
    mm = {}
    for m in masks:
        mm.setdefault(norm_stem(m), m)
    pairs = [(i, mm[norm_stem(i)]) for i in imgs if norm_stem(i) in mm]
    if len(pairs) < 0.5 * len(imgs) and len(imgs) == len(masks):
        warnings.warn("Name matching failed; pairing images and masks by sorted order.")
        pairs = list(zip(imgs, masks))
    print(f"{Path(img_dir).name}: {len(imgs)} images, {len(masks)} masks, {len(pairs)} pairs")
    return pairs

import hashlib

def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def drop_exact_duplicates(pairs, name):
    """Keep one record per byte-identical (image, mask) pair. Also reports images that appear
    with *different* masks (conflicting labels are kept; they act as label-noise averaging)."""
    seen, kept, dups, img_masks = {}, [], [], {}
    for ip, mp in pairs:
        hi, hm = sha256_file(ip), sha256_file(mp)
        img_masks.setdefault(hi, set()).add(hm)
        if (hi, hm) in seen:
            dups.append((Path(ip).name, Path(seen[(hi, hm)]).name))
        else:
            seen[(hi, hm)] = ip; kept.append((ip, mp))
    conflicts = sum(len(v) > 1 for v in img_masks.values())
    print(f"{name}: {len(pairs)} pairs -> {len(kept)} unique "
          f"({len(dups)} exact duplicates removed, {conflicts} images with conflicting masks)")
    if dups:
        print("   e.g.", dups[:3])
    return kept, pd.DataFrame(dups, columns=["removed", "duplicate_of"])

In [3]:
if Path(CFG["INPUT_ROOT"]).exists():
    DIRS = find_dirs(CFG["INPUT_ROOT"])
    SAMPLE_SUB = next(Path(CFG["INPUT_ROOT"]).rglob("sample_submission.csv"), None)
    print({k: str(v) for k, v in DIRS.items()}, "\nsample:", SAMPLE_SUB)
    APO_PAIRS, apo_dups = drop_exact_duplicates(pair_images_masks(DIRS["apo_img"], DIRS["apo_mask"]), "apo")
    FASC_PAIRS, fasc_dups = drop_exact_duplicates(pair_images_masks(DIRS["fasc_img"], DIRS["fasc_mask"]), "fasc")
    pd.concat([apo_dups.assign(set="apo"), fasc_dups.assign(set="fasc")]).to_csv(WORK / "removed_duplicates.csv", index=False)
    TEST_FILES = list_images(DIRS["test"])
    print("test images:", len(TEST_FILES))
    if CFG["DEBUG_MAX_TRAIN"]:
        APO_PAIRS = APO_PAIRS[: CFG["DEBUG_MAX_TRAIN"]]
        FASC_PAIRS = FASC_PAIRS[: CFG["DEBUG_MAX_TRAIN"]]

{'apo_img': '/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data/apo_imgs_v1', 'apo_mask': '/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data/apo_masks_v1', 'fasc_img': '/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data/fasc_imgs_v1', 'fasc_mask': '/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data/fasc_masks_v1', 'test': '/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data/test_images_v2'} 
sample: /kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data/sample_submission.csv
apo_imgs_v1: 1048 images, 1048 masks, 1048 pairs
apo: 1048 pairs -> 1048 unique (0 exact duplicates removed, 19 images with conflicting masks)
fasc_imgs_v1: 2761 images, 2761 masks, 2761 pairs
fasc: 2761 pairs -> 2091 unique (670 exact duplicates removed, 0 images with conflicting masks)
   e.g. [('image_0797.tif', 'image_0371.tif'), ('image_

## 2. Field-of-view crop, duplicate grouping

In [4]:
def _longest_run(b):
    """(start, end_exclusive) of the longest True run in a 1-D bool array."""
    best, cur, start, bs = 0, 0, 0, 0
    for i, v in enumerate(b):
        if v:
            if cur == 0:
                start = i
            cur += 1
            if cur > best:
                best, bs = cur, start
        else:
            cur = 0
    return (bs, bs + best) if best > 0 else (0, len(b))

def fov_box(g):
    """Bounding box (x, y, w, h) of the B-mode image area inside a screenshot.
    Uses local texture (speckle) and row/column projections so side text and colour bars are ignored."""
    H, W = g.shape
    f = g.astype(np.float32)
    mu = cv2.blur(f, (15, 15))
    sd = np.sqrt(np.maximum(cv2.blur(f * f, (15, 15)) - mu * mu, 0))
    m = ((sd > 4) & (mu > 6)).astype(np.uint8) * 255
    m = cv2.medianBlur(m, 5) > 0
    r0, r1 = _longest_run(m.mean(1) > 0.25)
    c0, c1 = _longest_run(m[r0:r1].mean(0) > 0.6)
    r0, r1 = _longest_run(m[:, c0:c1].mean(1) > 0.6)
    c0, c1 = _longest_run(m[r0:r1].mean(0) > 0.6)
    w, h = c1 - c0, r1 - r0
    if w * h < 0.15 * H * W or w * h > 0.92 * H * W or w < 32 or h < 32:
        return (0, 0, W, H)
    e = 7  # the 15x15 texture window bleeds ~7 px past the true border
    return (int(c0 + e), int(r0 + e), int(w - 2 * e), int(h - 2 * e))

def crop(a, box):
    x, y, w, h = box
    return a[y:y + h, x:x + w]

def dhash(g, n=8):
    s = cv2.resize(g, (n + 1, n), interpolation=cv2.INTER_AREA).astype(np.int16)
    bits = (s[:, 1:] > s[:, :-1]).flatten()
    return np.packbits(bits).view(">u8")[0] if n == 8 else bits

def duplicate_groups(hashes, max_dist=5):
    """Union-find over perceptual hashes: exact and near duplicates share a group (keeps them out of val)."""
    h = np.asarray(hashes, dtype=np.uint64)
    n = len(h)
    parent = list(range(n))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    pairs = []
    for s in range(0, n, 256):  # chunked to keep memory small
        x = (h[s:s + 256, None] ^ h[None, :]).view(np.uint8).reshape(-1, n, 8)
        dist = np.unpackbits(x, axis=2).sum(2)
        ii, jj = np.nonzero(dist <= max_dist)
        pairs += [(i + s, j) for i, j in zip(ii, jj) if i + s < j]
    for i, j in pairs:
        a, b = find(i), find(j)
        if a != b:
            parent[b] = a
    return np.array([find(i) for i in range(n)])

def load_pairs(pairs, size, thin=False):
    """Read, FOV-crop and resize image/mask pairs into memory."""
    X = np.zeros((len(pairs), size, size), np.uint8)
    Y = np.zeros((len(pairs), size, size), np.uint8)
    hashes = []
    for k, (ip, mp) in enumerate(pairs):
        g, m = read_gray(ip), read_mask(mp)
        if m.shape != g.shape:
            m = cv2.resize(m, g.shape[::-1], interpolation=cv2.INTER_NEAREST)
        b = fov_box(g)
        g, m = crop(g, b), crop(m, b)
        X[k] = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA)
        mf = cv2.resize(m.astype(np.float32), (size, size), interpolation=cv2.INTER_LINEAR if thin else cv2.INTER_NEAREST)
        Y[k] = (mf > (0.2 if thin else 0.5)).astype(np.uint8)
        hashes.append(dhash(X[k]))
    groups = duplicate_groups(hashes)
    print(f"loaded {len(pairs)} pairs, {len(set(groups))} unique groups "
          f"({len(pairs) - len(set(groups))} near/exact duplicates)")
    return X, Y, groups

def group_split(groups, val_frac, seed, n_folds=1, fold=0):
    """Train/val indices with whole duplicate groups on one side. n_folds>1 -> group k-fold, fold `fold`."""
    ug = np.unique(groups)
    rng = np.random.default_rng(seed); rng.shuffle(ug)
    if n_folds > 1:
        val_g = set(np.array_split(ug, n_folds)[fold])
    else:
        val_g = set(ug[: max(1, int(round(val_frac * len(ug))))])
    va = np.array([g in val_g for g in groups])
    return np.nonzero(~va)[0], np.nonzero(va)[0]

## 3. Model, augmentation, training

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(CFG["SEED"])
print("device:", DEVICE)

class ConvBlock(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = ConvBlock(cin + cskip, cout)
    def forward(self, x, skip=None):
        size = skip.shape[-2:] if skip is not None else (x.shape[-2] * 2, x.shape[-1] * 2)
        x = F.interpolate(x, size=size, mode="bilinear", align_corners=False)
        if skip is not None:
            x = torch.cat([x, skip], 1)
        return self.conv(x)

class ResUNet(nn.Module):
    """U-Net with a torchvision ResNet-34 encoder; grayscale input in [0, 1]."""
    def __init__(self, pretrained=True):
        super().__init__()
        enc = None
        if pretrained:
            try:
                enc = torchvision.models.resnet34(weights=torchvision.models.ResNet34_Weights.IMAGENET1K_V1)
            except Exception as e:
                print("ImageNet weights unavailable, random init:", type(e).__name__)
        if enc is None:
            enc = torchvision.models.resnet34(weights=None)
        self.stem = nn.Sequential(enc.conv1, enc.bn1, enc.relu)
        self.pool = enc.maxpool
        self.l1, self.l2, self.l3, self.l4 = enc.layer1, enc.layer2, enc.layer3, enc.layer4
        self.u4, self.u3 = Up(512, 256, 256), Up(256, 128, 128)
        self.u2, self.u1, self.u0 = Up(128, 64, 64), Up(64, 64, 64), Up(64, 0, 32)
        self.head = nn.Conv2d(32, 1, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        x = (x.repeat(1, 3, 1, 1) - self.mean) / self.std
        x0 = self.stem(x)
        x1 = self.l1(self.pool(x0))
        x2 = self.l2(x1); x3 = self.l3(x2); x4 = self.l4(x3)
        d = self.u4(x4, x3); d = self.u3(d, x2); d = self.u2(d, x1); d = self.u1(d, x0); d = self.u0(d)
        return self.head(d)

def augment(im, m):
    S = im.shape[0]
    if np.random.rand() < 0.5:
        im, m = np.ascontiguousarray(im[:, ::-1]), np.ascontiguousarray(m[:, ::-1])
    if np.random.rand() < 0.8:  # rotation, anisotropic scale (device aspect differences), shift
        a = math.radians(np.random.uniform(-10, 10))
        sx, sy = np.random.uniform(0.85, 1.2, 2)
        A = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]]) @ np.diag([sx, sy])
        c = np.array([S / 2, S / 2]); t = np.random.uniform(-0.06, 0.06, 2) * S
        M = np.hstack([A, (c - A @ c + t)[:, None]]).astype(np.float32)
        im = cv2.warpAffine(im, M, (S, S), flags=cv2.INTER_LINEAR, borderValue=0)
        m = cv2.warpAffine(m, M, (S, S), flags=cv2.INTER_NEAREST, borderValue=0)
    f = im.astype(np.float32) / 255
    if np.random.rand() < 0.8:
        f = f * np.random.uniform(0.7, 1.3) + np.random.uniform(-0.1, 0.1)
    if np.random.rand() < 0.5:
        f = np.clip(f, 0, 1) ** np.random.uniform(0.7, 1.5)
    if np.random.rand() < 0.3:  # multiplicative speckle
        f = f * np.random.gamma(4.0, 0.25, f.shape).astype(np.float32)
    if np.random.rand() < 0.2:
        f = cv2.GaussianBlur(f, (0, 0), np.random.uniform(0.5, 1.5))
    return np.clip(f * 255, 0, 255).astype(np.uint8), m

class SegDS(torch.utils.data.Dataset):
    def __init__(self, X, Y, idx, aug, dilate=0):
        self.X, self.Y, self.idx, self.aug = X, Y, idx, aug
        self.ker = np.ones((dilate, dilate), np.uint8) if dilate > 1 else None
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        im, m = self.X[i], self.Y[i]
        if self.aug:
            im, m = augment(im, m)
        if self.ker is not None:
            m = cv2.dilate(m, self.ker)
        return (torch.from_numpy(np.ascontiguousarray(im)).float().div(255)[None],
                torch.from_numpy(np.ascontiguousarray(m)).float()[None])

def dice_bce(logits, y, pos_weight=None):
    bce = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
    p = torch.sigmoid(logits)
    inter = (p * y).sum((1, 2, 3)); den = p.sum((1, 2, 3)) + y.sum((1, 2, 3))
    return bce + (1 - (2 * inter + 1) / (den + 1)).mean()

def _worker_init(wid):
    np.random.seed(CFG["SEED"] + wid + 1000 * torch.initial_seed() % 2**16)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); inter = den = 0.0; loss = 0.0; n = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            lo = model(x)
        lo = lo.float(); loss += dice_bce(lo, y).item() * len(x); n += len(x)
        p = (torch.sigmoid(lo) > 0.5).float()
        inter += (p * y).sum().item(); den += (p.sum() + y.sum()).item()
    return loss / max(n, 1), 2 * inter / max(den, 1e-6)

def train_seg(name, X, Y, groups, epochs, dilate=0, pos_weight=None, fold=0):
    tr, va = group_split(groups, CFG["VAL_FRAC"], CFG["SEED"], CFG["N_FOLDS"], fold)
    name = f"{name}_f{fold}"
    print(f"[{name}] train {len(tr)}  val {len(va)}")
    kw = dict(num_workers=CFG["NUM_WORKERS"], pin_memory=DEVICE.type == "cuda", worker_init_fn=_worker_init)
    dl_tr = torch.utils.data.DataLoader(SegDS(X, Y, tr, True, dilate), batch_size=CFG["BATCH"], shuffle=True, drop_last=True, **kw)
    dl_va = torch.utils.data.DataLoader(SegDS(X, Y, va, False, dilate), batch_size=CFG["BATCH"], shuffle=False, **kw)
    model = ResUNet(CFG["PRETRAINED"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    total = epochs * max(len(dl_tr), 1); warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=DEVICE.type == "cuda")
    pw = torch.tensor([pos_weight], device=DEVICE) if pos_weight else None
    best, hist = -1.0, []
    for ep in range(epochs):
        model.train(); t0 = time.time(); run = 0.0
        for x, y in dl_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                lo = model(x)
            loss = dice_bce(lo.float(), y, pw)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
        vl, vd = evaluate(model, dl_va)
        hist.append((ep, run / max(len(dl_tr), 1), vl, vd))
        if vd > best:
            best = vd; torch.save(model.state_dict(), WORK / f"{name}_unet.pt")
        print(f"[{name}] ep {ep+1:02d}/{epochs}  train {hist[-1][1]:.4f}  val {vl:.4f}  dice {vd:.4f}  "
              f"best {best:.4f}  ({time.time()-t0:.0f}s)")
    model.load_state_dict(torch.load(WORK / f"{name}_unet.pt", map_location=DEVICE))
    return model.eval(), va, hist

def load_models(name):
    paths = sorted(WEIGHTS.glob(f"{name}_f*_unet.pt"))
    assert paths, f"no {name} weights in {WEIGHTS}"
    models = []
    for p in paths:
        m = ResUNet(pretrained=False).to(DEVICE)
        m.load_state_dict(torch.load(p, map_location=DEVICE)); models.append(m.eval())
    print(f"loaded {len(models)} {name} model(s)")
    return models

device: cuda


In [6]:
S = CFG["IMG_SIZE"]
if CFG["TRAIN"]:
    Xa, Ya, Ga = load_pairs(APO_PAIRS, S, thin=False)
    APO_MODELS = []
    for k in range(CFG["N_FOLDS"]):
        m, va, _ = train_seg("apo", Xa, Ya, Ga, CFG["APO_EPOCHS"], fold=k)
        APO_MODELS.append(m)
        if k == 0:
            apo_val = va
    Xf, Yf, Gf = load_pairs(FASC_PAIRS, S, thin=True)
    FASC_MODELS = []
    for k in range(CFG["N_FOLDS"]):
        m, va, _ = train_seg("fasc", Xf, Yf, Gf, CFG["FASC_EPOCHS"], dilate=CFG["FASC_DILATE"], pos_weight=3.0, fold=k)
        FASC_MODELS.append(m)
        if k == 0:
            fasc_val = va
else:
    APO_MODELS, FASC_MODELS = load_models("apo"), load_models("fasc")

[ WARN:0@1723.520] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@1723.520] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@1723.663] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@1723.663] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@1723.814] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@1723.814] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@1724.140] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@1724.140] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc6

loaded 1048 pairs, 979 unique groups (69 near/exact duplicates)
[apo_f0] train 889  val 159
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 234MB/s]


[apo_f0] ep 01/40  train 1.3794  val 1.1822  dice 0.6957  best 0.6957  (33s)
[apo_f0] ep 02/40  train 1.1461  val 1.0550  dice 0.7547  best 0.7547  (32s)
[apo_f0] ep 03/40  train 1.0068  val 0.9418  dice 0.7621  best 0.7621  (32s)
[apo_f0] ep 04/40  train 0.8740  val 0.7921  dice 0.7822  best 0.7822  (34s)
[apo_f0] ep 05/40  train 0.7444  val 0.6775  dice 0.7762  best 0.7822  (33s)
[apo_f0] ep 06/40  train 0.6257  val 0.5557  dice 0.7725  best 0.7822  (33s)
[apo_f0] ep 07/40  train 0.5330  val 0.4916  dice 0.7693  best 0.7822  (33s)
[apo_f0] ep 08/40  train 0.4787  val 0.4498  dice 0.7785  best 0.7822  (33s)
[apo_f0] ep 09/40  train 0.4325  val 0.3926  dice 0.7794  best 0.7822  (33s)
[apo_f0] ep 10/40  train 0.3985  val 0.3574  dice 0.7987  best 0.7987  (33s)
[apo_f0] ep 11/40  train 0.3835  val 0.3442  dice 0.7956  best 0.7987  (33s)
[apo_f0] ep 12/40  train 0.3587  val 0.3443  dice 0.7861  best 0.7987  (33s)
[apo_f0] ep 13/40  train 0.3378  val 0.3318  dice 0.7895  best 0.7987  (33s)

[ WARN:0@3263.537] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@3263.537] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@3264.174] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@3264.174] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@3264.415] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@3264.415] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc697) encountered
[ WARN:0@3264.526] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50838 (0xc696) encountered
[ WARN:0@3264.526] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 50839 (0xc6

loaded 2091 pairs, 1134 unique groups (957 near/exact duplicates)
[fasc_f0] train 1911  val 180
[fasc_f0] ep 01/30  train 1.4960  val 1.3348  dice 0.3408  best 0.3408  (70s)
[fasc_f0] ep 02/30  train 1.2297  val 1.1451  dice 0.3816  best 0.3816  (70s)
[fasc_f0] ep 03/30  train 1.0494  val 0.9638  dice 0.4834  best 0.4834  (69s)
[fasc_f0] ep 04/30  train 0.9081  val 0.8276  dice 0.4824  best 0.4834  (69s)
[fasc_f0] ep 05/30  train 0.7999  val 0.7727  dice 0.4805  best 0.4834  (69s)
[fasc_f0] ep 06/30  train 0.7227  val 0.7025  dice 0.4842  best 0.4842  (69s)
[fasc_f0] ep 07/30  train 0.6787  val 0.6659  dice 0.5097  best 0.5097  (69s)
[fasc_f0] ep 08/30  train 0.6474  val 0.6472  dice 0.5016  best 0.5097  (69s)
[fasc_f0] ep 09/30  train 0.6294  val 0.6229  dice 0.5171  best 0.5171  (70s)
[fasc_f0] ep 10/30  train 0.6190  val 0.6298  dice 0.5165  best 0.5171  (69s)
[fasc_f0] ep 11/30  train 0.6054  val 0.6363  dice 0.5084  best 0.5171  (69s)
[fasc_f0] ep 12/30  train 0.5982  val 0.6021  

## 4. Geometry: aponeurosis lines, fascicle segments, PA / FL / MT (in pixels)

In [7]:
def robust_line(xs, ys, iters=3):
    """y = a*x + b with iterative MAD outlier rejection."""
    xs, ys = np.asarray(xs, float), np.asarray(ys, float)
    a, b = np.polyfit(xs, ys, 1)
    for _ in range(iters):
        r = ys - (a * xs + b)
        s = 1.4826 * np.median(np.abs(r - np.median(r))) + 1e-6
        keep = np.abs(r) < 2.5 * s + 1.0
        if keep.sum() < 10:
            break
        a, b = np.polyfit(xs[keep], ys[keep], 1)
    return float(a), float(b)

def _border_points(mask, side):
    """Per-column inner border of a band: side='bottom' -> max row, 'top' -> min row."""
    cols = np.nonzero(mask.any(0))[0]
    if len(cols) == 0:
        return cols, cols
    if side == "bottom":
        ys = mask.shape[0] - 1 - np.argmax(mask[::-1, cols], axis=0)
    elif side == "center":
        sub = mask[:, cols].astype(np.float64)
        ys = (sub * np.arange(mask.shape[0])[:, None]).sum(0) / sub.sum(0)
    else:
        ys = np.argmax(mask[:, cols], axis=0)
    return cols, ys

def find_aponeuroses(apo_prob, fasc_prob=None, thr=0.5, edge="inner"):
    """Returns dict with superficial/deep inner-border lines (a, b) or None."""
    H, W = apo_prob.shape
    m = (apo_prob > thr).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(m)
    comps = []
    for i in range(1, n):
        if st[i, cv2.CC_STAT_AREA] < 0.0005 * H * W or st[i, cv2.CC_STAT_WIDTH] < 0.08 * W:
            continue
        ys, xs = np.nonzero(lab == i)
        a, b = robust_line(xs, ys) if len(xs) >= 10 else (0.0, float(ys.mean()))
        comps.append(dict(ids=[i], a=a, b=b, xmin=xs.min(), xmax=xs.max()))
    # merge fragments of the same aponeurosis (close centre lines)
    comps.sort(key=lambda c: c["a"] * W / 2 + c["b"])
    merged = []
    for c in comps:
        if merged:
            p = merged[-1]
            xm = W / 2
            if abs((c["a"] * xm + c["b"]) - (p["a"] * xm + p["b"])) < 0.04 * H:
                p["ids"] += c["ids"]; p["xmin"] = min(p["xmin"], c["xmin"]); p["xmax"] = max(p["xmax"], c["xmax"])
                continue
        merged.append(dict(c))
    bands = []
    for c in merged:
        bm = np.isin(lab, c["ids"])
        cover = bm.any(0).mean()
        if cover < 0.25:
            continue
        bands.append(dict(mask=bm, cover=cover, yc=c["a"] * W / 2 + c["b"]))
    if len(bands) < 2:
        return None
    bands.sort(key=lambda d: d["yc"])
    # choose the adjacent pair that encloses the most fascicle signal (the muscle of interest)
    best, best_s = None, -1e9
    for k in range(len(bands) - 1):
        top, bot = bands[k], bands[k + 1]
        if bot["yc"] - top["yc"] < 0.06 * H:
            continue
        s = 0.5 * min(top["cover"], bot["cover"])
        if fasc_prob is not None:
            y0, y1 = int(max(0, top["yc"])), int(min(H, bot["yc"]))
            s += float(fasc_prob[y0:y1].mean()) * 10 if y1 > y0 else 0
        if s > best_s:
            best, best_s = (top, bot), s
    if best is None:
        return None
    if edge == "center":
        xs, ys = _border_points(best[0]["mask"], "center"); xd, yd = _border_points(best[1]["mask"], "center")
    else:
        xs, ys = _border_points(best[0]["mask"], "bottom"); xd, yd = _border_points(best[1]["mask"], "top")
    sup, deep = robust_line(xs, ys), robust_line(xd, yd)
    return dict(sup=sup, deep=deep, sup_cover=best[0]["cover"], deep_cover=best[1]["cover"])

def fascicle_segments(fasc_prob, apo=None, thr=0.35, min_len=20, margin_frac=0.03):
    """Elongated components of the fascicle probability map (inside the muscle if apo given)."""
    H, W = fasc_prob.shape
    p = fasc_prob.copy()
    if apo is not None:
        X = np.arange(W)[None, :]; Yg = np.arange(H)[:, None]
        (a_s, b_s), (a_d, b_d) = apo["sup"], apo["deep"]
        mt = np.abs((a_d - a_s) * W / 2 + (b_d - b_s))
        mg = margin_frac * mt
        inside = (Yg > a_s * X + b_s + mg) & (Yg < a_d * X + b_d - mg)
        p[~inside] = 0
    m = (p > thr).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(m)
    segs = []
    for i in range(1, n):
        x0, y0, w, h, area = st[i]
        if area < 8:
            continue
        sub = lab[y0:y0 + h, x0:x0 + w] == i
        ys, xs = np.nonzero(sub)
        pts = np.stack([xs + x0, ys + y0], 1).astype(float)
        c = pts.mean(0); C = np.cov((pts - c).T) + 1e-6 * np.eye(2)
        ev, evec = np.linalg.eigh(C)
        d = evec[:, 1]
        if d[0] < 0:
            d = -d
        proj = (pts - c) @ d
        length = proj.max() - proj.min()
        elong = math.sqrt(ev[1] / max(ev[0], 1e-6))
        if length < min_len or elong < 3:
            continue
        wgt = length * float(fasc_prob[ys + y0, xs + x0].mean())
        segs.append(dict(cx=c[0], cy=c[1], dx=d[0], dy=d[1], length=length, w=wgt))
    return segs

def structure_tensor_angle(img, region, sigma=6.0):
    """Dominant line orientation (radians, image coords, in (-pi/2, pi/2]) in a region."""
    f = img.astype(np.float32)
    gx = cv2.Sobel(f, cv2.CV_32F, 1, 0, ksize=5); gy = cv2.Sobel(f, cv2.CV_32F, 0, 1, ksize=5)
    jxx = cv2.GaussianBlur(gx * gx, (0, 0), sigma); jyy = cv2.GaussianBlur(gy * gy, (0, 0), sigma)
    jxy = cv2.GaussianBlur(gx * gy, (0, 0), sigma)
    coh = np.sqrt((jxx - jyy) ** 2 + 4 * jxy ** 2) / (jxx + jyy + 1e-6)
    w = coh * region
    # gradient orientation (doubled angle), averaged, then rotated 90deg -> line orientation
    c2 = (w * (jxx - jyy)).sum(); s2 = (w * 2 * jxy).sum()
    theta_g = 0.5 * math.atan2(s2, c2)
    return theta_g + math.pi / 2

def weighted_median(v, w):
    v, w = np.asarray(v, float), np.asarray(w, float)
    o = np.argsort(v); v, w = v[o], w[o]
    cw = np.cumsum(w)
    return float(v[np.searchsorted(cw, 0.5 * cw[-1])])

def signed_angle_to(u, f):
    """Signed angle (deg) from direction u to direction f (image coordinates)."""
    return math.degrees(math.atan2(u[0] * f[1] - u[1] * f[0], u[0] * f[0] + u[1] * f[1]))

def dominant_angle(segs, u):
    """Weighted-median signed angle (deg) of segments relative to direction u, using only the
    dominant fascicle orientation. Returns (angle or nan, n_used)."""
    ang, wts = [], []
    for s in segs:
        f = np.array([s["dx"], s["dy"]])
        if f @ u < 0:
            f = -f
        ang.append(signed_angle_to(u, f)); wts.append(s["w"])
    ang, wts = np.array(ang), np.array(wts)
    if len(ang) < 2:
        return np.nan, 0
    sign = 1.0 if wts[ang > 0].sum() >= wts[ang < 0].sum() else -1.0
    keep = (np.sign(ang) == sign) & (np.abs(ang) > 2) & (np.abs(ang) < 70)
    if keep.sum() < 2:
        return np.nan, 0
    return sign * weighted_median(np.abs(ang[keep]), wts[keep]), int(keep.sum())

def architecture_px(img, apo_prob, fasc_prob, cfg=CFG):
    """PA (deg), FL (px), MT (px) for one FOV crop + diagnostics."""
    H, W = img.shape
    out = dict(pa=np.nan, fl_px=np.nan, mt_px=np.nan, n_segs=0, method="fail", apo=None, segs=[], fl_lines=[])
    apo = find_aponeuroses(apo_prob, fasc_prob, cfg["APO_THR"], cfg["MT_EDGE"])
    if apo is None:
        return out
    out["apo"] = apo
    (a_s, b_s), (a_d, b_d) = apo["sup"], apo["deep"]
    xs_anchor = np.array([0.25, 0.5, 0.75]) * W
    # --- MT: perpendicular distance, both directions averaged, at 3 positions
    mts = []
    for x in xs_anchor:
        yd = a_d * x + b_d; ys_ = a_s * x + b_s
        d1 = abs(a_s * x - yd + b_s) / math.hypot(a_s, 1)
        d2 = abs(a_d * x - ys_ + b_d) / math.hypot(a_d, 1)
        mts.append(0.5 * (d1 + d2))
    out["mt_px"] = float(np.mean(mts))
    # --- PA: fascicle segments relative to the deep aponeurosis
    u = np.array([1.0, a_d]) / math.hypot(1.0, a_d)
    segs = fascicle_segments(fasc_prob, apo, cfg["FASC_THR"], max(10, cfg["MIN_FASC_LEN_FRAC"] * W))
    out["segs"] = segs
    sa, n_used = dominant_angle(segs, u)
    if np.isfinite(sa):
        out["n_segs"] = n_used; out["method"] = "segments"
    if np.isnan(sa):  # fallback: structure tensor inside the muscle
        X = np.arange(W)[None, :]; Yg = np.arange(H)[:, None]
        mg = 0.15 * out["mt_px"]  # stay clear of the bright aponeurosis edges
        region = ((Yg > a_s * X + b_s + mg) & (Yg < a_d * X + b_d - mg)).astype(np.float32)
        region *= (0.2 + fasc_prob)
        if region.sum() > 100:
            th = structure_tensor_angle(img, region, sigma=3.0)
            f = np.array([math.cos(th), math.sin(th)])
            if f @ u < 0:
                f = -f
            sa = signed_angle_to(u, f)
            out["method"] = "structure_tensor"
    if np.isnan(sa) or abs(sa) < 1:
        return out
    out["pa"] = abs(sa)
    # --- FL: from deep anchors along the fascicle direction to the superficial line
    r = math.radians(sa)
    f = np.array([u[0] * math.cos(r) - u[1] * math.sin(r), u[0] * math.sin(r) + u[1] * math.cos(r)])
    if f[1] > 0:  # walk upwards (towards the superficial aponeurosis)
        f = -f
    fls = []
    for x in xs_anchor:
        q = np.array([x, a_d * x + b_d])
        den = f[1] - a_s * f[0]
        if abs(den) < 1e-9:
            continue
        t = (a_s * q[0] + b_s - q[1]) / den
        if t > 0:
            fls.append(t); out["fl_lines"].append((q, q + t * f))
    if fls:
        out["fl_px"] = float(np.mean(fls))
    return out

## 5. Pixel → mm calibration from the on-screen depth scale

In [8]:
def detect_ticks(g, box):
    """Find a column of regularly spaced short marks (depth ruler) outside the FOV interior.
    Returns dict(spacing_px, n, x) or None."""
    H, W = g.shape
    x, y, w, h = box
    bright = (g > 90).astype(np.uint8)
    pad = 6
    if w < W or h < H:
        bright[y + pad:y + h - pad, x + pad:x + w - pad] = 0
    n, lab, st, cen = cv2.connectedComponentsWithStats(bright, connectivity=8)
    cand = []
    for i in range(1, n):
        cx, cy, cw, ch, a = st[i]
        if 2 <= cw <= 40 and 1 <= ch <= 6 and a >= 2 and cw >= 0.6 * ch:
            cand.append((cx, cx + cw, cen[i][0], cen[i][1]))
    if len(cand) < 5:
        return None
    cand = np.array(cand)
    best = None
    for key in (0, 1, 2):  # align by left edge, right edge, centre
        o = np.argsort(cand[:, key]); c = cand[o]
        splits = np.nonzero(np.diff(c[:, key]) > 3)[0] + 1
        for grp in np.split(c, splits):
            if len(grp) < 5:
                continue
            ys = np.sort(grp[:, 3])
            ys = ys[np.concatenate([[True], np.diff(ys) > 3])]
            if len(ys) < 5:
                continue
            d = np.diff(ys); dm = np.median(d)
            if dm < 8:
                continue
            k = np.round((ys - ys[0]) / dm)  # tolerate missing ticks
            res = ys - ys[0] - k * dm
            inl = np.abs(res) < 0.12 * dm
            if inl.sum() < 5:
                continue
            A = np.stack([k[inl], np.ones(inl.sum())], 1)
            sp = float(np.linalg.lstsq(A, ys[inl], rcond=None)[0][0])
            span = ys[inl].max() - ys[inl].min()
            ov = max(0, min(ys[inl].max(), y + h) - max(ys[inl].min(), y)) / max(h, 1)
            if ov < 0.4:
                continue
            score = inl.sum() + 0.01 * span
            if best is None or score > best["score"]:
                best = dict(spacing_px=sp, n=int(inl.sum()), x=float(grp[:, 2].mean()), score=score)
    return best

def resolve_calibration(meta, cfg=CFG):
    """meta: DataFrame with W,H,fov_h,tick_spacing,mt_px. Adds px_per_mm, calib_src, calib_group."""
    meta = meta.copy()
    meta["size_key"] = meta["W"].astype(int).astype(str) + "x" + meta["H"].astype(int).astype(str)
    meta["calib_group"] = meta["size_key"] + "|" + meta["tick_spacing"].apply(
        lambda s: "noticks" if not np.isfinite(s) else f"s{int(round(s / 3.0) * 3)}")
    meta["px_per_mm"] = np.nan; meta["mm_per_tick"] = np.nan; meta["calib_src"] = ""
    rows = []
    for gk, d in meta.groupby("calib_group"):
        sk = d["size_key"].iloc[0]
        if sk in cfg["CALIB_PX_PER_MM"]:
            meta.loc[d.index, "px_per_mm"] = cfg["CALIB_PX_PER_MM"][sk]; meta.loc[d.index, "calib_src"] = "override_px"
            continue
        if gk.endswith("noticks"):
            continue
        if sk in cfg["CALIB_MM_PER_TICK"]:
            mm, src = cfg["CALIB_MM_PER_TICK"][sk], "override_tick"
        else:
            best, mm = 1e9, None
            for cmm in cfg["TICK_MM_CANDIDATES"]:
                ppm = d["tick_spacing"] / cmm
                mt = np.nanmedian(d["mt_px"] / ppm); depth = np.nanmedian(d["fov_h"] / ppm)
                if not np.isfinite(mt):
                    mt = cfg["MT_PRIOR_MM"] * 3  # no MT info -> rely on depth only
                sc = abs(math.log(mt / cfg["MT_PRIOR_MM"]))
                sc += 0 if 20 <= depth <= 120 else 1.0
                sc += 0 if cfg["MT_RANGE"][0] <= mt <= cfg["MT_RANGE"][1] else 1.0
                if sc < best:
                    best, mm = sc, cmm
            src = "ticks"
        meta.loc[d.index, "mm_per_tick"] = mm
        meta.loc[d.index, "px_per_mm"] = d["tick_spacing"] / mm
        meta.loc[d.index, "calib_src"] = src
    # images without ticks: borrow from same image size, else default depth
    for i in meta.index[meta["px_per_mm"].isna()]:
        same = meta[(meta["size_key"] == meta.at[i, "size_key"]) & meta["px_per_mm"].notna()
                    & meta["calib_src"].str.startswith(("ticks", "override"))]
        if len(same):
            meta.at[i, "px_per_mm"] = same["px_per_mm"].median(); meta.at[i, "calib_src"] = "borrowed"
        else:
            meta.at[i, "px_per_mm"] = meta.at[i, "fov_h"] / cfg["DEFAULT_DEPTH_MM"]; meta.at[i, "calib_src"] = "default_depth"
    meta["implied_depth_mm"] = meta["fov_h"] / meta["px_per_mm"]
    return meta

## 6. Inference on the test set

In [9]:
@torch.no_grad()
def predict_prob(models, g):
    """Mean probability over a list of models (fold ensemble) and optional flip TTA, at the input's size."""
    models = models if isinstance(models, (list, tuple)) else [models]
    h, w = g.shape
    x = cv2.resize(g, (S, S), interpolation=cv2.INTER_AREA)
    t = torch.from_numpy(x).float().div(255)[None, None].to(DEVICE)
    if CFG["TTA"]:
        t = torch.cat([t, t.flip(-1)])
    acc = 0
    for model in models:
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            p = torch.sigmoid(model(t).float())
        acc = acc + ((p[0:1] + p[1:2].flip(-1)) / 2 if CFG["TTA"] else p)
    p = acc / len(models)
    return cv2.resize(p[0, 0].cpu().numpy(), (w, h), interpolation=cv2.INTER_LINEAR)

def analyse_image(path):
    g = read_gray(path)
    H, W = g.shape
    box = fov_box(g)
    c = crop(g, box)
    ap, fp = predict_prob(APO_MODELS, c), predict_prob(FASC_MODELS, c)
    res = architecture_px(c, ap, fp)
    tk = detect_ticks(g, box)
    res.update(path=str(path), W=W, H=H, box=box, fov_h=box[3],
               tick_spacing=tk["spacing_px"] if tk else np.nan, n_ticks=tk["n"] if tk else 0)
    return res

### Official UMUD score + internal validation proxy
The training set has masks but no PA/FL/MT labels, so the proxy compares the geometry read-out on
*predicted* vs *ground-truth* masks of held-out (duplicate-free, group-split) fold-0 images:
PA error in degrees (÷6 = score units) and relative MT error (×20 mm ÷ 3 ≈ score units for a 20 mm muscle).

In [10]:
TOL = {"pa_deg": 6.0, "fl_mm": 12.0, "mt_mm": 3.0}   # from the official UMUD score notebook

def umud_score(solution, submission, id_col="image_id"):
    """Primary UMUD Score: mean over targets of MAE / tolerance (lower is better), plus a per-target report."""
    m = solution.merge(submission, on=id_col, suffixes=("_true", "_pred"))
    assert len(m) == len(solution), "submission is missing ids"
    rep = {c: dict(mae=float(np.mean(np.abs(m[c + "_pred"] - m[c + "_true"]))),
                   bias=float(np.mean(m[c + "_pred"] - m[c + "_true"]))) for c in TOL}
    for c in rep:
        rep[c]["norm"] = rep[c]["mae"] / TOL[c]
    return float(np.mean([rep[c]["norm"] for c in TOL])), pd.DataFrame(rep).T

def validation_proxy(max_n=150):
    rows = []
    for i in apo_val[:max_n]:
        g = Xa[i]
        gt = find_aponeuroses(Ya[i].astype(np.float32), None, 0.5, CFG["MT_EDGE"])
        pr = find_aponeuroses(predict_prob(APO_MODELS[:1], g), None, CFG["APO_THR"], CFG["MT_EDGE"])
        if gt is None:
            continue
        mt = lambda a: abs((a["deep"][0] - a["sup"][0]) * S / 2 + a["deep"][1] - a["sup"][1]) / math.hypot(1, a["deep"][0])
        rows.append(dict(kind="mt", true_val=mt(gt), pred_val=mt(pr) if pr else np.nan))
    u = np.array([1.0, 0.0])  # fascicle angle vs. image horizontal (apo masks not available for this set)
    for i in fasc_val[:max_n]:
        a_gt, _ = dominant_angle(fascicle_segments(Yf[i].astype(np.float32), None, 0.5, 10), u)
        a_pr, _ = dominant_angle(fascicle_segments(predict_prob(FASC_MODELS[:1], Xf[i]), None, CFG["FASC_THR"], 10), u)
        if np.isfinite(a_gt):
            rows.append(dict(kind="pa", true_val=abs(a_gt), pred_val=abs(a_pr) if np.isfinite(a_pr) else np.nan))
    # column names chosen to avoid DataFrame methods (e.g. `.gt`); always use df["col"] access
    v = pd.DataFrame(rows, columns=["kind", "true_val", "pred_val"])
    mt_df, pa_df = v[v["kind"] == "mt"], v[v["kind"] == "pa"]
    mt_rel = (np.abs(mt_df["pred_val"] - mt_df["true_val"]) / mt_df["true_val"]).median()
    pa_mae = np.abs(pa_df["pred_val"] - pa_df["true_val"]).mean()
    print(f"MT: {len(mt_df)} imgs, failures {mt_df['pred_val'].isna().sum()}, median rel. error {100*mt_rel:.1f}% "
          f"(~{mt_rel*20/3:.3f} score units at 20 mm)")
    print(f"PA: {len(pa_df)} imgs, failures {pa_df['pred_val'].isna().sum()}, MAE {pa_mae:.2f} deg "
          f"({pa_mae/6:.3f} score units)")
    v.to_csv(WORK / "validation_proxy.csv", index=False)
    return v

if CFG["TRAIN"]:
    try:  # a diagnostic must never stop the run before the submission is written
        VAL_PROXY = validation_proxy()
    except Exception as e:
        import traceback; traceback.print_exc()
        print("validation proxy failed (submission continues):", e)

MT: 141 imgs, failures 1, median rel. error 0.8% (~0.054 score units at 20 mm)
PA: 150 imgs, failures 0, MAE 1.40 deg (0.234 score units)


In [11]:
# The sample submission is only a format example (2 rows, ids like "IMG_00001.tif").
# The real submission needs one row per test image, so ids come from the test folder,
# written in the same style as the sample (with or without the file extension).
sample_ids = pd.read_csv(SAMPLE_SUB)["image_id"].astype(str).tolist() if SAMPLE_SUB is not None else []
use_ext = any(Path(s).suffix.lower() in IMG_EXT for s in sample_ids) if sample_ids else True
sub_ids = sorted({(p.name if use_ext else p.stem) for p in TEST_FILES})
by_stem = {p.stem: p for p in TEST_FILES}
by_name = {p.name: p for p in TEST_FILES}
def id_to_path(i):
    return by_name.get(i) or by_stem.get(i) or by_stem.get(Path(i).stem)
missing = [i for i in sub_ids if id_to_path(i) is None]
print(f"sample format: {sample_ids[:2]} -> {'with' if use_ext else 'without'} extension")
print(f"{len(sub_ids)} test ids (expected 309), {len(missing)} without an image file", missing[:5])
if len(sub_ids) != 309:
    warnings.warn(f"Found {len(sub_ids)} test images, the competition states 309 — check the test folder.")
not_in_test = [s for s in sample_ids if s not in sub_ids]
if not_in_test:
    warnings.warn(f"Sample ids not matched to test files: {not_in_test} — check id format.")

RESULTS = {}
t0 = time.time()
for k, i in enumerate(sub_ids):
    p = id_to_path(i)
    if p is not None:
        RESULTS[i] = analyse_image(p)
    if (k + 1) % 50 == 0:
        print(f"{k+1}/{len(sub_ids)}  {time.time()-t0:.0f}s")

meta = pd.DataFrame([dict(image_id=i, W=r["W"], H=r["H"], fov_h=r["fov_h"], tick_spacing=r["tick_spacing"],
                          n_ticks=r["n_ticks"], pa=r["pa"], fl_px=r["fl_px"], mt_px=r["mt_px"],
                          n_segs=r["n_segs"], method=r["method"]) for i, r in RESULTS.items()])
meta = resolve_calibration(meta)
calib_table = (meta.groupby("calib_group")
               .agg(n=("image_id", "size"), src=("calib_src", "first"), tick_px=("tick_spacing", "median"),
                    mm_per_tick=("mm_per_tick", "first"), px_per_mm=("px_per_mm", "median"),
                    depth_mm=("implied_depth_mm", "median"), mt_px=("mt_px", "median")))
calib_table["mt_mm"] = calib_table["mt_px"] / calib_table["px_per_mm"]
print("Check 'depth_mm' against the depth label shown on the scanner screen:")
print(calib_table.round(2).to_string())
print(meta["method"].value_counts().to_string())

KeyError: 'image_id'

## 7. Temporal smoothing over near-identical frames, plausibility limits, submission

In [ ]:
def frame_neighbors(meta, thr, k):
    ids = list(meta["image_id"])
    vecs = []
    for i in ids:
        r = RESULTS[i]
        c = crop(read_gray(r["path"]), r["box"])
        v = cv2.resize(c, (96, 64), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
        v -= v.mean(); v /= (np.linalg.norm(v) + 1e-6)
        vecs.append(v)
    V = np.stack(vecs); Sim = V @ V.T
    size = meta["size_key"].values
    nb = {}
    for a in range(len(ids)):
        cand = [b for b in np.argsort(-Sim[a]) if b != a and Sim[a, b] > thr and size[a] == size[b]][:k]
        nb[ids[a]] = [ids[b] for b in cand]
    return nb

meta["pa_deg"] = meta["pa"]
meta["fl_mm"] = meta["fl_px"] / meta["px_per_mm"]
meta["mt_mm"] = meta["mt_px"] / meta["px_per_mm"]
lims = {"pa_deg": CFG["PA_RANGE"], "fl_mm": CFG["FL_RANGE"], "mt_mm": CFG["MT_RANGE"]}
for c, (lo, hi) in lims.items():
    bad = (meta[c] < lo * 0.6) | (meta[c] > hi * 1.5)   # wildly implausible -> treat as failure
    meta.loc[bad, c] = np.nan

# Low-confidence PA -> shrink toward the group median (MAE-optimal under uncertainty). FL is rescaled with
# the same geometry (FL ∝ 1/sin PA for a given thickness), so PA and FL stay consistent.
grp_pa = meta.groupby("calib_group")["pa_deg"].transform("median").fillna(meta["pa_deg"].median())
w = meta["n_segs"] / (meta["n_segs"] + CFG["PA_SHRINK_K"])
w = w.where(meta["method"] != "structure_tensor", 0.5)
ok = meta["pa_deg"].notna() & grp_pa.notna()
pa_new = w * meta["pa_deg"] + (1 - w) * grp_pa
meta.loc[ok, "fl_mm"] = meta.loc[ok, "fl_mm"] * np.sin(np.radians(meta.loc[ok, "pa_deg"])) / np.sin(np.radians(pa_new[ok]))
meta.loc[ok, "pa_deg"] = pa_new[ok]

NB = frame_neighbors(meta,CFG["SEQ_SIM_THR"], CFG["SEQ_MAX_NEIGHBORS"])
print("images with >=1 temporal neighbour:", sum(len(v) > 0 for v in NB.values()))
mi = meta.set_index("image_id")
sm = mi[list(lims)].copy()
for i, nbs in NB.items():
    if nbs:
        sm.loc[i] = mi.loc[[i] + nbs, list(lims)].median(skipna=True).values
# fill failures: group median, then global median, then mid-range prior
for c, (lo, hi) in lims.items():
    sm[c] = sm[c].fillna(mi.groupby("calib_group")[c].transform("median"))
    sm[c] = sm[c].fillna(sm[c].median()).fillna((lo + hi) / 2).clip(lo, hi)
print("failed (filled) values:", {c: int(mi[c].isna().sum()) for c in lims})

sub = pd.DataFrame({"image_id": sub_ids})
sub = sub.merge(sm.reset_index()[["image_id", "pa_deg", "fl_mm", "mt_mm"]], on="image_id", how="left")
for c, (lo, hi) in lims.items():
    sub[c] = sub[c].fillna(sm[c].median() if len(sm) else (lo + hi) / 2).round(3)
assert sub["image_id"].is_unique and len(sub) == len(sub_ids)
sub.to_csv(WORK / "submission.csv", index=False)
meta.to_csv(WORK / "predictions_debug.csv", index=False)
print(sub.describe().round(2))
if CFG["LOCAL_LABELS"] and Path(CFG["LOCAL_LABELS"]).exists():
    lab = pd.read_csv(CFG["LOCAL_LABELS"]); lab["image_id"] = lab["image_id"].astype(str)
    sc, rep = umud_score(lab, sub[sub.image_id.isin(lab.image_id)])
    print(f"UMUD score on {len(lab)} locally labelled images: {sc:.4f}"); print(rep.round(3))
sub.head()

## 8. Visual QC — check aponeurosis lines, fascicle direction and scale on a few test images

In [ ]:
def draw_qc(i, ax):
    r = RESULTS[i]
    c = crop(read_gray(r["path"]), r["box"])
    H, W = c.shape
    ax.imshow(c, cmap="gray"); ax.set_xlim(0, W); ax.set_ylim(H, 0)
    if r["apo"] is not None:
        xx = np.array([0, W])
        for key, col in (("sup", "cyan"), ("deep", "magenta")):
            a, b = r["apo"][key]; ax.plot(xx, a * xx + b, col, lw=1.5)
    for s in r["segs"]:
        L = s["length"] / 2
        ax.plot([s["cx"] - L * s["dx"], s["cx"] + L * s["dx"]], [s["cy"] - L * s["dy"], s["cy"] + L * s["dy"]], "y-", lw=1)
    for q, e in r["fl_lines"]:
        ax.plot([q[0], e[0]], [q[1], e[1]], "lime", lw=1.2)
    row = sub.set_index("image_id").loc[i]
    ax.set_title(f"{i}  PA {row.pa_deg:.1f}°  FL {row.fl_mm:.0f}  MT {row.mt_mm:.1f} mm", fontsize=9)
    ax.axis("off")

show = list(RESULTS)[:: max(1, len(RESULTS) // 8)][:8]
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax in axes.ravel():
    ax.axis("off")
for i, ax in zip(show, axes.ravel()):
    draw_qc(i, ax)
plt.tight_layout(); plt.savefig(WORK / "qc_examples.png", dpi=80); plt.show()